# 16 — Plan a common-coverage geolocation pilot

The 39-row sample contains some MODIS–VIIRS candidate edges among rows selected independently by quarter/platform/status. This notebook recomputes those edges, chooses the smallest time separation (distance as tie-breaker), reads the **exact geolocation input filenames named in the fire-product files**, and asks CMR for those exact granules. This avoids substituting a newer geolocation processing revision.

NASA describes Terra MOD03 as the companion 1 km product with geodetic coordinates for each pixel center ([MOD03 product page](https://ladsweb.modaps.eosdis.nasa.gov/missions-and-measurements/products/MOD03/)). NASA describes VIIRS `VJ203IMG`/related I-band geolocation as 375 m imagery-resolution geolocation ([LAADS VIIRS L1 product guide](https://ladsweb.modaps.eosdis.nasa.gov/missions-and-measurements/science-domain/viirs-L0-L1/)). The fire-mask array can then be sampled at the geolocation grid cell nearest the other sensor's detection coordinate to classify clear-land/fire versus cloud/unknown/unprocessed opportunity.

This stage only builds a metadata plan. It does not download the geolocation granules or claim that a candidate is a shared physical fire.

In [1]:
from pathlib import Path
import sys,json,hashlib
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'data').exists() and (ROOT.parent/'data').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from src.firewatch.association import find_candidate_links
from src.firewatch.geolocation_manifest import extract_geolocation_reference,query_exact_geolocation_manifest
SAMPLE=ROOT/'data/reference/validation_sample_2025.csv'
SAMPLE_META=ROOT/'data/reference/validation_sample_2025.metadata.json'
L2_DIR=ROOT/'data/reference/granules/validation_2025'
DOWNLOAD_META=L2_DIR/'download_manifest.json'
sample=pd.read_csv(SAMPLE)
sample=sample.loc[sample.selection_status.eq('matched')].copy()
meta=json.loads(SAMPLE_META.read_text(encoding='utf-8'))
assert len(sample)==39 and sample.observation_id.is_unique
print('Loaded',len(sample),'time/CMR-matched sample observations; sample seed:',meta['seed'])

Loaded 39 time/CMR-matched sample observations; sample seed: firewatch-2025-pixel-validation-v1


In [2]:
# Recompute candidate edges among the selected rows and choose a compact real pair.
obs=sample.rename(columns={'latitude':'latitude_num','longitude':'longitude_num','observation_time_utc':'observation_datetime_utc'}).copy()
obs['validation_status']='valid'; obs['observation_datetime_utc']=pd.to_datetime(obs.observation_datetime_utc,utc=True)
links=find_candidate_links(obs,max_distance_km=1.0,max_time_delta_hours=3.0)
cross=links.loc[links.link_family.eq('MODIS-VIIRS')].sort_values(['time_difference_minutes','distance_km','candidate_link_id'],kind='stable')
assert len(cross)>0
edge=cross.iloc[0]
display(cross[['candidate_link_id','left_satellite','right_satellite','left_datetime_utc','right_datetime_utc','distance_km','time_difference_minutes']])
print(f'Selected edge: {edge.distance_km:.3f} km apart, {edge.time_difference_minutes:.0f} minutes apart.')

,candidate_link_id,left_satellite,right_satellite,left_datetime_utc,right_datetime_utc,distance_km,time_difference_minutes
1,22c2d022b93cdf2be214b18204f14a68c497a768f41a09...,Aqua,N21,2025-07-06 13:48:00+00:00,2025-07-06 12:30:00+00:00,0.253693,78.0
0,ee5f12d3b0d0356149be1a4eb9b357c37c74bc5ee22fe0...,Aqua,N20,2025-07-06 13:48:00+00:00,2025-07-06 11:39:00+00:00,0.889563,129.0


Selected edge: 0.254 km apart, 78 minutes apart.


In [3]:
# Extract product processing inputs from the chosen L2 source files.
source_rows=[]
for side in ['left','right']:
    observation_id=str(edge[f'{side}_observation_id'])
    row=sample.loc[sample.observation_id.eq(observation_id)].iloc[0]
    l2_path=L2_DIR/Path(row.data_url).name
    ref=extract_geolocation_reference(l2_path)
    source_rows.append({
        **ref,'candidate_link_id':edge.candidate_link_id,
        'observation_id':observation_id,'satellite':row.satellite,
        'l2_product':row.short_name,'l2_granule':row.producer_granule_id,
        'observation_time_utc':row.observation_time_utc,
        'observation_latitude':float(row.latitude),'observation_longitude':float(row.longitude),
        'edge_distance_km':float(edge.distance_km),
        'edge_time_difference_minutes':float(edge.time_difference_minutes),
    })
references=pd.DataFrame(source_rows)
assert references.geolocation_granule_id.is_unique
refs=references.to_dict(orient='records')
cmr_rows,cmr_metadata=query_exact_geolocation_manifest(refs)
coverage_plan=pd.DataFrame(cmr_rows)
assert set(coverage_plan.geolocation_granule_id)==set(references.geolocation_granule_id)
assert coverage_plan.data_url.notna().all()
cmr_total=float(coverage_plan.granule_size_cmr_mb.sum())
display(coverage_plan[['satellite','l2_product','l2_granule','geolocation_short_name','cmr_version','cmr_collection_version','geolocation_granule_id','granule_size_cmr_mb','time_start_utc','time_end_utc']])
print('Exact companion granules:',len(coverage_plan),'| CMR size-field sum:',round(cmr_total,3))

,satellite,l2_product,l2_granule,geolocation_short_name,cmr_version,cmr_collection_version,geolocation_granule_id,granule_size_cmr_mb,time_start_utc,time_end_utc
0,Aqua,MYD14,MYD14.A2025187.1345.061.2025188110859,MYD03,6.1,6.1,MYD03.A2025187.1345.061.2025188145533.hdf,32.973021,2025-07-06T13:45:00.000Z,2025-07-06T13:50:00.000Z
1,N21,VJ214IMG,VJ214IMG.A2025187.1230.002.2025327203213,VJ203IMG,2.1,2.1,VJ203IMG.A2025187.1230.021.2025187181751.nc,184.669895,2025-07-06T12:30:00.000Z,2025-07-06T12:36:00.000Z


Exact companion granules: 2 | CMR size-field sum: 217.643


In [4]:
# Keep the download plan limited to one MODIS–VIIRS candidate pair.
PLAN_OUT=ROOT/'data/reference/geolocation_coverage_plan_2025.csv'
META_OUT=ROOT/'data/reference/geolocation_coverage_plan_2025.metadata.json'
coverage_plan.to_csv(PLAN_OUT,index=False)
metadata={
 'selection':'Shortest time separation among MODIS–VIIRS links within 1 km / 3 h among the 39 selected source rows; distance breaks ties.',
 'candidate_link_id':str(edge.candidate_link_id),
 'edge_distance_km':float(edge.distance_km),'edge_time_difference_minutes':float(edge.time_difference_minutes),
 'l2_sample_csv_sha256':hashlib.sha256(SAMPLE.read_bytes()).hexdigest(),
 'l2_download_manifest_sha256':hashlib.sha256(DOWNLOAD_META.read_bytes()).hexdigest(),
 'l2_files':references.l2_granule.tolist(),
 'geolocation_files':coverage_plan.geolocation_granule_id.tolist(),
 'geolocation_count':len(coverage_plan),'cmr_size_field_sum':cmr_total,
 'cmr_query_metadata':cmr_metadata,
 'granule_files_downloaded':0,
 'planned_cap_mib':250,
 'limitations':['A candidate edge is not event identity.','The geolocation input names are exact processing lineage, but swath coverage must still be measured from arrays.','The L2 fire mask is algorithm output and not independent ground truth.','Cross-time clear pixels within a 78-minute sample interval do not prove continuous burning or absence.']
}
META_OUT.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
print('Saved exact geolocation plan:',PLAN_OUT.relative_to(ROOT))
print(f'CMR estimate {cmr_total:.3f} MiB; bounded transfer cap {metadata["planned_cap_mib"]} MiB; no files downloaded.')

Saved exact geolocation plan: data\reference\geolocation_coverage_plan_2025.csv
CMR estimate 217.643 MiB; bounded transfer cap 250 MiB; no files downloaded.


## Coverage analysis after retrieval

Once the two exact geolocation files are available, compare each geolocation latitude/longitude grid with the paired L2 fire-mask grid. For each direction in the selected pair, find the nearest valid geolocation cell to the other source's FIRMS coordinate, then inspect its mask class. On land, class 5 is clear/non-fire land and classes 7–9 are fire detections; cloud (4), unknown (6), missing/not-processed (0–2) are not clear-land observation opportunities. Preserve water separately. This is a pixel-level overpass/opportunity check for one selected pair only, not a general statement about fires that were or were not present.